# Step 3 - Train: XGBoost / LightGBM / random forest x feature sets, compared on validation

Builds the training and validation tables once from step 2 (identical rows stored in two files are counted once; DoS and Normal are sampled to 1M unique rows; the **test split is not read**), then trains every experiment in `configs/experiments.yaml` in order:

| experiment | model | features |
|---|---|---|
| xgb_full | XGBoost | all 28 |
| xgb_portable | XGBoost | no TTL, ports as type/service instead of numbers |
| xgb_no_ttl | XGBoost | all but TTL |
| lgbm_full / lgbm_portable | LightGBM | full / portable |
| rf_portable | random forest | portable |
| xgb_portable_fine | XGBoost | portable, 17 raw labels instead of 6 classes |

Every class gets the same total weight; models stop early when validation loss stops improving.

* **Runtime:** a **T4 GPU** makes the XGBoost experiments much faster (*Runtime -> Change runtime type -> T4 GPU*); CPU works too, it just takes longer (roughly 2-3 h for everything instead of about 1 h).
* **Disconnected?** Reconnect and run cells 1, 2 and 4 again: finished experiments are skipped and the running one continues from its last checkpoint (every 25 rounds).
* **Only some experiments:** `!python -u -m gids train --only xgb_full,xgb_portable`
* **Results:** copy what cell 6 prints into the chat (it can be pasted after only some experiments are done).


In [ ]:
#@title 1. Settings
REPO_URL = "https://github.com/madhumithakk01/ds.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
ZIP_PATH = "/content/drive/MyDrive/Ds/GothamDataset2025.zip"  # "Ds" = shortcut to the shared folder  #@param {type:"string"}
WORK_DIR = "/content/drive/MyDrive/gotham_ids"  #@param {type:"string"}

In [ ]:
#@title 2. Mount Drive, get the latest code, check the environment
import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")

REPO_DIR = "/content/ds"
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(REPO_DIR, "requirements.txt")], check=True)

os.chdir(REPO_DIR)
if not os.path.exists(ZIP_PATH):
    folder = os.path.dirname(ZIP_PATH)
    print(f"ZIP_PATH not found: {ZIP_PATH}")
    if os.path.isdir(folder):
        print(f"Contents of {folder}:")
        for name in sorted(os.listdir(folder))[:50]:
            print("  ", name)
    print("Set ZIP_PATH in cell 1 to the zip shown above (the code will also search My Drive for it).")
os.environ["GIDS_WORK_DIR"] = WORK_DIR
os.environ["GIDS_ZIP_PATH"] = ZIP_PATH
print(subprocess.run(["git", "log", "-1", "--format=code version %h (%cd)"], capture_output=True, text=True).stdout)
!python -m gids env

In [ ]:
#@title 3. Self-test on a small synthetic archive (about 2 min; includes a crash + resume of training)
!python -m gids selftest


In [ ]:
#@title 4. Train every experiment (resumable: after a disconnect run cells 1, 2 and 4 again)
!python -u -m gids train


In [ ]:
#@title 5. Report
from IPython.display import Markdown, display
display(Markdown(open(f"{WORK_DIR}/reports/train/TRAIN.md").read()))


In [ ]:
#@title 6. Plain text of the report (copy all of it into the chat)
print(open(f"{WORK_DIR}/reports/train/TRAIN.md").read())


### If something goes wrong
* `status` lists every experiment: done (with its validation macro-F1), rounds so far, or not started.
* `trace --errors` prints warnings and errors of the latest run with full tracebacks.
* Out of memory: lower the sample, e.g. `!python -u -m gids --set dataset.max_rows_per_class=500000 train` (this builds a new dataset folder).
* Models, metrics, confusion matrices and feature importance of each experiment are in `MyDrive/gotham_ids/models/<experiment>-<hash>/`.


In [ ]:
!python -m gids status
!python -m gids trace --errors --tail 30
